# Quick Start

We'll start by generating some fake data with seasonality in.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydemetra as jd
from pandas.plotting import register_matplotlib_converters

# Create a sample monthly time series
index = pd.period_range("2000-01", periods=240, freq="M")
rng = np.random.default_rng(42)

# 1. Your original Random Walk
rw = np.cumsum(rng.normal(size=240)) + 100

# 2. Add Sine Wave (Cycle length = 12 months)
# amplitude controls how "tall" the waves are
amplitude = 5
seasonal_cycle = amplitude * np.sin(2 * np.pi * np.arange(240) / 12)

ts = pd.Series(rw + seasonal_cycle, index=index)
# Create a mapping for each month (1-12)
# Let's boost Dec (12) and Jan (1) and tank July (7)
seasonal_map = {1: 2.0, 7: -5.0, 12: 4.0}
effects = [seasonal_map.get(m, 0) for m in index.month]

ts_sharp = pd.Series(rw + effects, index=index)

## Running a pre-built specification

Let's create a default X-13 specification and run seasonal adjustment on it, then extract the key components.

In [ ]:
spec = jd.x13_spec("rsa5c")
result = jd.x13(ts_sharp, spec)

sa = result["result"]["final"]["d11final"]  # seasonally adjusted series
trend = result["result"]["final"]["d12final"]  # trend
seasonal = result["result"]["final"]["d16"]  # seasonal component
irregular = result["result"]["final"]["d13final"]  # irregular component

In [ ]:
sa

And here's what the results look like:

In [ ]:
# | echo: false


plt.style.use("plot_style.txt")


register_matplotlib_converters()


def plot_lines(ts, sa, trend):
    fig, ax = plt.subplots()
    lw = 2
    ts.plot(ax=ax, label="Original", lw=lw, ls="solid")
    sa.plot(ax=ax, label="Seasonally adjusted", lw=lw)
    trend.plot(ax=ax, label="Trend", lw=lw, alpha=0.9)
    ax.legend(frameon=False)
    plt.show()


plot_lines(ts_sharp, sa, trend)

## Creating a specification

This time we'll begin with a specification and amend it a bit

In [ ]:
spec = jd.x13_spec("rsa5c")
# Force log transformation (no automatic detection)
spec["regarima"]["transform"]["fn"] = "LOG"

# Disable transitory component (TC) outlier detection
spec["regarima"]["outlier"]["outliers"] = [
    o for o in spec["regarima"]["outlier"]["outliers"] if o["type"] != "TC"
]

# Disable trading day regressors (suitable for quarterly data)
spec["regarima"]["regression"]["td"]["td"] = "TD_NONE"
spec["regarima"]["regression"]["td"]["auto"] = "AUTO_NO"

result = jd.x13(ts_sharp, spec)

In [ ]:
sa = result["result"]["final"]["d11final"]  # seasonally adjusted series
trend = result["result"]["final"]["d12final"]  # trend
seasonal = result["result"]["final"]["d16"]  # seasonal component
irregular = result["result"]["final"]["d13final"]  # irregular component

How do the results look?

In [ ]:
sa.head(5)